In [75]:
import bs4
import requests as r
from xml.etree import ElementTree as ET
import pandas as pd
from pandas import DataFrame
META_URL = 'https://courses.rice.edu/courses/!SWKSCAT.info'
BASE_URL = 'https://courses.rice.edu/courses/courses/!SWKSCAT.cat'

def get_term_codes() -> DataFrame:    
    req = r.get(f"{META_URL}?action=TERMS") 
    terms = ET.fromstring(req.text).findall('TERM')
    df = []

    for term in terms:
        # ignore quadmesters since they are only for the Glasscock school, not undergraduate
        if not "Quadmester" in term.find('OPT').tail:
            df.append({
                'term': term.find('OPT').tail,
                'code': term.attrib.get('code')
            })

    return DataFrame(df)

def get_subject_codes(term_code: str) -> DataFrame:
    req = r.get(f"{META_URL}?action=SUBJECTS&term={term_code}") 
    subjects = ET.fromstring(req.text).findall('SUBJECT')
    df = []

    for subject in subjects:
        df.append({
            'subject': subject.find('OPT').tail,
            'code': subject.attrib.get('code')
        })

    return DataFrame(df)

def get_course_syllabus(term_code: str, crn: str):
    req = r.get(f"{META_URL}?action=SYLLABUS&term={term_code}&crn={crn}") 
    res = ET.fromstring(req.text)
    if res.attrib.get('has-syllabus') != 'yes':
        return None
    return res.attrib.get('doc-url')

def get_schools(term_code: str) -> DataFrame:
    req = r.get(f"{META_URL}?action=SCHOOLS&term={term_code}") 
    schools = ET.fromstring(req.text).findall('SCHOOL')
    df = []

    for school in schools:
        df.append({
            'school': school.find('OPT').tail,
            'code': school.attrib.get('code')
        })

    return DataFrame(df)

In [ ]:
def get_course_info(term_code: str, school_code: str) -> DataFrame:
    courses = []
    req = r.get(f"{BASE_URL}?p_action=QUERY&p_term={term_code}&p_school={school_code}")
    parser = bs4.BeautifulSoup(req.text, 'html.parser')
    rows = parser.find('tbody').find_all('tr')
    rows[0].find_all('td')

    for row in parser.find('tbody').find_all('tr'):
        cells = row.find_all('td')
        if len(cells) != 7:
            raise ValueError(f"Expected 7 cells per row, got {len(cells)}")
        courses.append({
            'crn': cells[0].text,
            'crs': cells[1].text,
            'title': cells[3].text, #ignore "part of term" entry
            'instructors': cells[4].text,
            'meeting_times': cells[5].text,
            'credits': cells[6].text,
        })

    return DataFrame(courses)

def get_all_courses(term_code: str, export_to_csv = False) -> DataFrame:
    schools = get_schools(term_code)
    all_courses = []

    for school_code in schools['code']:
        all_courses.append(get_course_info(term_code, school_code))
    
    result = pd.concat(all_courses, ignore_index=True)
    if export_to_csv:
        result.to_csv(f'courses_{term_code}.csv', index=False)

    return result

df = get_all_courses('202620', export_to_csv=True)